# Biological information comparison

**Purpose:** Compare condition-and-biology prediction with and without additional gait information.

**Required inputs:** Prepared Baček HDF5; biological-data CSV; 04 and 05 results ZIPs.

**Outputs:** Selected models and matched prediction records.

**Usage:** Run after 04 and 05. Files are read from the notebook’s current working directory unless specified in its configuration.


In [ ]:
# 1 — Imports, exact input files, fixed settings and helpers
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import io
import json
import os
import platform
import sys
import time
import zipfile

import h5py
import numpy as np
import scipy
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

H5 = Path('bacek_clean_common_angles_metabolic_dataset.h5')
BIO_CSV = Path('Bacek2023_subject_biological_data.csv')
REFERENCE04 = Path('04_Bacek_Foundational_Predictive_Models_results.zip')
REFERENCE05 = Path('05_Bacek_Incremental_Gait_Information_results.zip')
OUT = Path('06_Bacek_Biological_Information_Comparison_results')
ARCHIVE = Path('06_Bacek_Biological_Information_Comparison_results.zip')
SOURCE_SHA = 'c73ce561e94de52f1dc86128829c43ef536c4cf6b5f4aa9af1ea060409221fa3'
CHANNELS = ('hip_left', 'hip_right', 'knee_left', 'knee_right',
            'ankle_left', 'ankle_right', 'pelvis_sagittal',
            'pelvis_frontal', 'pelvis_transverse')
ALPHAS = (0.01, 0.1, 1.0, 10.0, 100.0, 1000.0)
PCA_COUNTS = (5, 10, 20, 40)
SPEEDS = (1.1, 0.8, 0.4)
CADENCES = (0.90, 0.95, 1.00, 1.10, 1.20)
OUTER_FOLDS = {
    1: ('S08', 'S10', 'S16', 'S21'),
    2: ('S11', 'S13', 'S17', 'S20'),
    3: ('S01', 'S02', 'S05', 'S18'),
    4: ('S03', 'S07', 'S12', 'S14', 'S19'),
    5: ('S04', 'S06', 'S09', 'S15'),
}
EXPECTED_COUNTS = (55, 53, 56, 65, 55)
FOLD_OF = {s: fold for fold, people in OUTER_FOLDS.items() for s in people}


def check(condition, message):
    if not bool(condition):
        raise AssertionError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(8*1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def read_csv_zip(archive, filename):
    with archive.open(filename) as file:
        return list(csv.DictReader(io.TextIOWrapper(file, encoding='utf-8')))


def write_csv(path, rows, fields):
    with path.open('w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def rmse(actual, predicted):
    errors = np.asarray(predicted, dtype=np.float64)-np.asarray(actual, dtype=np.float64)
    return float(np.sqrt(np.mean(errors*errors, dtype=np.float64)))


def metrics(actual, predicted):
    actual = np.asarray(actual, dtype=np.float64)
    predicted = np.asarray(predicted, dtype=np.float64)
    err = predicted - actual
    denom = float(np.sum((actual-actual.mean())**2, dtype=np.float64))
    return {
        'n_trials': len(actual), 'RMSE_W_per_kg': rmse(actual, predicted),
        'MAE_W_per_kg': float(np.mean(np.abs(err), dtype=np.float64)),
        'bias_W_per_kg': float(np.mean(err, dtype=np.float64)),
        'R2': float(1.0-np.sum(err**2, dtype=np.float64)/denom) if denom>0 else None,
    }

print('Python:', platform.python_version(), '| NumPy:', np.__version__,
      '| scikit-learn:', sklearn.__version__)

In [ ]:
# 2 — Read and cross-check the verified Notebook 04 / 05 reference evidence
for path in (REFERENCE04, REFERENCE05):
    check(path.is_file(), f'Required previous results archive missing: {path.name}')
sha04, sha05 = sha256_file(REFERENCE04), sha256_file(REFERENCE05)
with zipfile.ZipFile(REFERENCE04) as z4:
    summary04 = json.loads(z4.read('run_summary.json'))
    ledger04 = read_csv_zip(z4, 'eligible_trial_ledger.csv')
    inner04 = read_csv_zip(z4, 'inner_fold_manifest.csv')
    previous04 = read_csv_zip(z4, 'held_out_predictions.csv')
    checks04 = read_csv_zip(z4, 'verification_checks.csv')
with zipfile.ZipFile(REFERENCE05) as z5:
    summary05 = json.loads(z5.read('run_summary.json'))
    ledger05 = read_csv_zip(z5, 'eligible_trial_ledger.csv')
    inner05 = read_csv_zip(z5, 'inner_fold_manifest.csv')
    previous05 = read_csv_zip(z5, 'held_out_predictions.csv')
    checks05 = read_csv_zip(z5, 'verification_checks.csv')
    previous_metrics05 = read_csv_zip(z5, 'pooled_metrics.csv')

check(summary04['status']=='PASS' and summary05['status']=='PASS',
      'Previous notebook reports unsuccessful verification')
check(all(r['status']=='PASS' for r in checks04+checks05),
      'Failed check in a previous results archive')
check(summary04['source_sha256']==SOURCE_SHA==summary05['source_sha256'],
      'Source version differs across notebooks')
check(summary05['reference_notebook04_zip_sha256']==sha04,
      'Notebook 05 is not linked to this exact Notebook 04 ZIP')
check(summary04['total_primary_trials']==284 and summary05['n_primary_trials']==284,
      'Prior eligible population mismatch')
check(len(ledger04)==len(ledger05)==284 and len(inner04)==len(inner05)==20,
      'Previous ledger or inner manifest incomplete')
check(inner04 == inner05, 'Notebook 05 did not preserve exact Notebook 04 inner folds')

prior_ledger = {r['trial_uid']:r for r in ledger04}
check(len(prior_ledger)==284, 'Duplicate UID in Notebook 04')
check({r['trial_uid'] for r in ledger05} == set(prior_ledger),
      'Notebook 05 and Notebook 04 trial UIDs differ')
for r in ledger05:
    old = prior_ledger[r['trial_uid']]
    for field in ('subject_id','trial_id'):
        check(r[field]==old[field], f'Prior ledger identity mismatch: {r["trial_uid"]}')
    for field in ('outer_fold','retained_cycles','condition_index'):
        check(int(r[field])==int(old[field]), f'Prior ledger numeric mismatch: {r["trial_uid"]}')
    for field in ('speed_m_s','relative_cadence','observed_W_per_kg'):
        check(float(r[field])==float(old[field]), f'Prior ledger value mismatch: {r["trial_uid"]}')

prior_predictions = {}
for label in ('C','C+G'):
    records = [r for r in previous05 if r['procedure']==label]
    table = {r['trial_uid']:r for r in records}
    check(len(records)==len(table)==284 and set(table)==set(prior_ledger),
          f'Notebook 05 {label} prediction coverage mismatch')
    prior_predictions[label] = table
    for uid, p in table.items():
        original=prior_ledger[uid]
        check(p['subject_id']==original['subject_id'] and
              p['trial_id']==original['trial_id'] and
              int(p['outer_fold'])==int(original['outer_fold']) and
              float(p['observed_W_per_kg'])==float(original['observed_W_per_kg']) and
              np.isfinite(float(p['predicted_W_per_kg'])),
              f'Invalid prior {label} prediction: {uid}')

prior_B2 = {r['trial_uid']:r for r in previous04 if r['procedure']=='B2'}
check(len(prior_B2)==284, 'Notebook 04 B2 coverage mismatch')
check(all(prior_B2[uid]['predicted_W_per_kg']==prior_predictions['C'][uid]['predicted_W_per_kg']
          for uid in prior_ledger), 'C predictions changed between Notebook 04 and 05')
print('Prior evidence accepted: 284 matched C / C+G predictions; exact 20 saved inner folds')

In [ ]:
# 3 — Verify companion biological CSV and the 21-person age/height join
check(BIO_CSV.is_file(), f'Biological CSV not found: {BIO_CSV.name}')
bio_sha = sha256_file(BIO_CSV)
with BIO_CSV.open('r', encoding='utf-8-sig', newline='') as file:
    reader = csv.DictReader(file)
    check(reader.fieldnames is not None and
          {'subject_id','age_years','height_m'}.issubset(reader.fieldnames),
          'Biological CSV is missing subject_id, age_years or height_m')
    biological_rows = list(reader)
check(len(biological_rows)==21, 'Expected exactly 21 participant biological records')
biological = {}
join_manifest = []
for record in biological_rows:
    subject = record['subject_id'].strip()
    check(subject in FOLD_OF and subject not in biological,
          f'Missing, duplicated or unexpected biological participant: {subject}')
    try:
        age = float(record['age_years'])
        height = float(record['height_m'])
    except (TypeError, ValueError) as exc:
        raise AssertionError(f'Invalid biological numbers for {subject}') from exc
    check(np.isfinite(age) and age>0 and np.isfinite(height) and height>0,
          f'Nonfinite/nonpositive biological input for {subject}')
    if 'subject_number' in record and record['subject_number']:
        check(int(record['subject_number'])==int(subject[1:]),
              f'Biological subject_number inconsistent for {subject}')
    biological[subject] = (age, height)
    join_manifest.append({'subject_id':subject, 'join_status':'PASS',
                          'age_present_and_positive':True,
                          'height_present_and_positive':True})
check(set(biological)==set(FOLD_OF), 'Biological join does not cover 21 participants')
check(len({r['subject_id'] for r in join_manifest})==21, 'Duplicate biological join key')
# Other columns (weight, BMI, sex, preferred cadence etc.) must never become inputs.
print('Verified biological source: 21 unique profiles; age/height only')

In [ ]:
# 4 — Reconstruct the 284 trial-level C+B, C+B+W and C+B+D inputs

def features_from_trial(trial):
    means=[]
    n_cycles=None
    for channel in CHANNELS:
        check(channel in trial, f'Missing gait channel: {channel}')
        angles=np.asarray(trial[channel][()],dtype=np.float64)
        check(angles.ndim==2 and angles.shape[1]==101 and np.isfinite(angles).all(),
              f'Malformed or nonfinite angle channel: {channel}')
        if n_cycles is None:
            n_cycles=angles.shape[0]
        check(angles.shape[0]==n_cycles, 'Mismatched cycle counts across channels')
        means.append(angles.mean(axis=0,dtype=np.float64))
    waveforms=np.stack(means)
    descriptors=[]
    for channel in waveforms:
        for q in range(4):
            descriptors.append(float(np.trapezoid(channel[q*25:q*25+26],dx=1.0)/25.0))
        descriptors.append(float(np.std(channel,ddof=0)))
        descriptors.append(float(np.max(channel)-np.min(channel)))
    return waveforms.reshape(909), np.asarray(descriptors,dtype=np.float64), n_cycles

check(H5.is_file(), f'Corrected HDF5 file missing: {H5.name}')
source_sha=sha256_file(H5)
check(source_sha==SOURCE_SHA, 'HDF5 SHA-256 mismatch — stop execution')
rows, wrows, drows, brows = [],[],[],[]
with h5py.File(H5,'r') as h5:
    check(set(h5['subjects'])==set(FOLD_OF), 'HDF5 participant IDs differ')
    for subject in sorted(h5['subjects']):
        for trial_id in sorted(h5['subjects'][subject]['trials']):
            if not (trial_id.startswith('T') and trial_id[1:].isdigit() and
                    1<=int(trial_id[1:])<=15):
                continue
            trial=h5['subjects'][subject]['trials'][trial_id]
            uid=f'{subject}_{trial_id}'
            w,d,n=features_from_trial(trial)
            for attr in ('valid','comparison_eligible','unrestricted_comparison_eligible'):
                check(attr in trial.attrs and
                      np.asarray(trial.attrs[attr]).reshape(-1)[0]==1,
                      f'{uid}: invalid source flag {attr}')
            target=np.asarray(trial['metcost_mean'][()])
            check(target.size==1 and np.issubdtype(target.dtype,np.number) and
                  np.isrealobj(target), f'{uid}: invalid scalar target')
            outcome=float(target.reshape(-1)[0])
            check(np.isfinite(outcome) and outcome>0, f'{uid}: invalid metabolic target')
            original=np.asarray(trial['original_cycle_number'][()]).reshape(-1)
            check(len(original)==n and np.isfinite(original).all() and
                  np.all(original==np.floor(original)) and np.all(np.diff(original)>0),
                  f'{uid}: invalid retained-cycle indices')
            if uid=='S06_T11':
                check(n==85, 'S06_T11 should contain 85 retained cycles')
                continue
            check(n>=100, f'{uid}: unexpected primary eligibility')
            condition=int(trial_id[1:])-1
            speed_block, cadence_index=divmod(condition,5)
            row={'trial_uid':uid, 'subject_id':subject, 'trial_id':trial_id,
                 'outer_fold':FOLD_OF[subject], 'retained_cycles':n,
                 'speed_m_s':SPEEDS[speed_block],
                 'relative_cadence':CADENCES[cadence_index],
                 'condition_index':condition, 'observed_W_per_kg':outcome}
            check(uid in prior_ledger, f'Unexpected primary record {uid}')
            old=prior_ledger[uid]
            for field in ('subject_id','trial_id'):
                check(row[field]==old[field],f'Notebook 04 mismatch for {uid}: {field}')
            for field in ('outer_fold','retained_cycles','condition_index'):
                check(int(row[field])==int(old[field]),f'Notebook 04 mismatch for {uid}: {field}')
            for field in ('speed_m_s','relative_cadence','observed_W_per_kg'):
                check(float(row[field])==float(old[field]),
                      f'Notebook 04 mismatch for {uid}: {field}')
            rows.append(row)
            wrows.append(w)
            drows.append(d)
            brows.append(biological[subject])

check(len(rows)==len({r['trial_uid'] for r in rows})==284,
      'Expected exactly 284 unique eligible trials')
check(sum(r['retained_cycles'] for r in rows)==60757, 'Primary retained-cycle total mismatch')
check([sum(r['outer_fold']==fold for r in rows) for fold in OUTER_FOLDS]
      ==list(EXPECTED_COUNTS), 'Primary outer-fold trial counts mismatch')
check(set(r['trial_uid'] for r in rows)==set(prior_ledger),
      'Primary trial identity coverage differs from Notebook 04')

Xc=np.eye(15,dtype=np.float64)[[r['condition_index'] for r in rows]]
Xb=np.asarray(brows,dtype=np.float64)
Xw=np.stack(wrows)
Xd=np.stack(drows)
Xcb=np.column_stack((Xc,Xb))
Xcbw=np.column_stack((Xc,Xb,Xw))
Xcbd=np.column_stack((Xc,Xb,Xd))
y=np.asarray([r['observed_W_per_kg'] for r in rows],dtype=np.float64)
groups=np.asarray([r['subject_id'] for r in rows])
uids=np.asarray([r['trial_uid'] for r in rows])
check(Xcb.shape==(284,17) and Xcbw.shape==(284,926) and
      Xcbd.shape==(284,71), 'Biological/gait design matrix shape mismatch')
check(all(np.isfinite(matrix).all() for matrix in (Xb,Xw,Xd,Xcb,Xcbw,Xcbd)),
      'Nonfinite constructed predictor')
check(np.all(Xc.sum(axis=1)==1), 'Invalid one-hot condition contract')
check(all(np.array_equal(Xb[groups==s],np.repeat([biological[s]],sum(groups==s),axis=0))
          for s in FOLD_OF), 'Age/height are not constant and correctly joined within person')
print('Verified and joined 284 trials / 21 participants; predictor shapes:',
      'C+B',Xcb.shape,'C+B+W',Xcbw.shape,'C+B+D',Xcbd.shape)

## Training-partition preprocessing and estimation


In [ ]:
# 5 — Exact C+B and C+B+G candidates, transformations and tie rules

def candidate_grid():
    cb=[{'procedure':'C+B','representation':'NONE','alpha':alpha,
         'n_components':None,'n_features':17} for alpha in ALPHAS]
    cbg=[]
    for representation,dimension in (('W-ALL9',909),('D-ALL9',54)):
        for alpha in ALPHAS:
            cbg.append({'procedure':'C+B+G','representation':representation,
                        'alpha':alpha,'n_components':None,'n_features':17+dimension})
    for count in PCA_COUNTS:
        for alpha in ALPHAS:
            cbg.append({'procedure':'C+B+G','representation':'P-ALL9',
                        'alpha':alpha,'n_components':count,'n_features':17+count})
    return cb,cbg


def candidate_matrix(candidate):
    if candidate['procedure']=='C+B':
        return Xcb
    return Xcbd if candidate['representation']=='D-ALL9' else Xcbw


def candidate_pipeline(candidate):
    blocks=[('conditions','passthrough',slice(0,15)),
            ('biology',StandardScaler(),slice(15,17))]
    if candidate['procedure']=='C+B+G':
        steps=[('scale',StandardScaler())]
        if candidate['representation']=='P-ALL9':
            steps.append(('pca',PCA(n_components=candidate['n_components'],
                                    whiten=False,random_state=2026)))
        blocks.append(('gait',Pipeline(steps),slice(17,None)))
    features=ColumnTransformer(blocks,remainder='drop')
    return Pipeline([('features',features),
                     ('ridge',Ridge(alpha=candidate['alpha'],fit_intercept=True))])


def choose(scores,procedure):
    if procedure=='C+B':
        return min(scores,key=lambda r:(r['inner_pooled_RMSE_W_per_kg'],r['alpha']))
    return min(scores,key=lambda r:(r['inner_pooled_RMSE_W_per_kg'],r['n_features'],
                                    r['representation'],r['n_components'] or 0,r['alpha']))


def assert_fitted_preprocessing(model,candidate,x_train):
    preprocessing=model.named_steps['features']
    # C indicators are copied without standardisation, dimensionality reduction or selection.
    transformed=preprocessing.transform(x_train)
    check(np.array_equal(transformed[:,:15],x_train[:,:15]),
          'Condition indicators were modified')
    bio_scaler=preprocessing.named_transformers_['biology']
    check(bio_scaler.n_features_in_==2 and
          np.allclose(bio_scaler.mean_,x_train[:,15:17].mean(axis=0),rtol=1e-12,atol=1e-12),
          'Biological scaler fitted outside the training partition')
    check(np.allclose(bio_scaler.var_,x_train[:,15:17].var(axis=0),rtol=1e-12,atol=1e-12),
          'Biological variance differs from training partition')
    if candidate['procedure']=='C+B+G':
        gait=preprocessing.named_transformers_['gait']
        check(np.allclose(gait.named_steps['scale'].mean_,
                          x_train[:,17:].mean(axis=0),rtol=1e-12,atol=1e-12),
              'Gait scaler fitted outside training partition')
        if candidate['representation']=='P-ALL9':
            pca=gait.named_steps['pca']
            check(pca.n_components_==candidate['n_components'] and
                  pca.n_samples_==len(x_train), 'PCA fit boundary or dimensions invalid')
    check(transformed.shape[1]==candidate['n_features'],
          'Transformed model input dimensionality mismatch')

CB_CANDIDATES,CBG_CANDIDATES=candidate_grid()
check(len(CB_CANDIDATES)==6 and len(CBG_CANDIDATES)==36,
      'E2.2 candidate counts must be 6 and 36')
check({c['n_features'] for c in CBG_CANDIDATES}=={22,27,37,57,71,926},
      'Unexpected C+B+G input dimensions')
check(len({(c['representation'],c['alpha'],c['n_components'])
           for c in CBG_CANDIDATES})==36, 'Repeated C+B+G candidate')

# Tiny synthetic fit: verify unscaled indicators and separate biology/gait fitted blocks.
rng=np.random.default_rng(2026)
x_demo=np.column_stack((np.eye(15)[np.arange(12)%15],
                        rng.normal(size=(12,2)),rng.normal(size=(12,909))))
y_demo=rng.normal(size=12)
for candidate,x_sample in ((CB_CANDIDATES[0],x_demo[:,:17]),
                           (CBG_CANDIDATES[0],x_demo),
                           (next(c for c in CBG_CANDIDATES
                                 if c['representation']=='P-ALL9' and c['n_components']==5),x_demo)):
    demo=candidate_pipeline(candidate)
    demo.fit(x_sample,y_demo)
    assert_fitted_preprocessing(demo,candidate,x_sample)
print('Synthetic pipeline check passed; candidate counts: C+B=6, C+B+G=36')

In [ ]:
# 6 — Reuse the exact 20 inner folds; fit and select the two complete procedures
started_at=datetime.now(timezone.utc).isoformat()
t0=time.perf_counter()
inner_scores,pooled_scores,selections,new_predictions,timings=[],[],[],[],[]
verified_inner=[]

for fold,test_people in OUTER_FOLDS.items():
    test_idx=np.flatnonzero(np.isin(groups,test_people))
    dev_idx=np.flatnonzero(~np.isin(groups,test_people))
    check(len(test_idx)==EXPECTED_COUNTS[fold-1], 'Outer test trial count mismatch')
    check(set(groups[test_idx]).isdisjoint(groups[dev_idx]),
          'Outer test people occur in development')
    saved=sorted((r for r in inner04 if int(r['outer_fold'])==fold),
                 key=lambda r:int(r['inner_fold']))
    check(len(saved)==4, f'Expected four saved inner folds for outer fold {fold}')
    inner_indices=[]
    validation_rows=[]
    for inner_number,record in enumerate(saved,1):
        train_people=set(record['train_participants'].split(';'))
        valid_people=set(record['validation_participants'].split(';'))
        expected_dev=set(FOLD_OF)-set(test_people)
        train_idx=np.flatnonzero(np.isin(groups,list(train_people)))
        valid_idx=np.flatnonzero(np.isin(groups,list(valid_people)))
        check(int(record['inner_fold'])==inner_number and
              train_people.isdisjoint(valid_people) and
              train_people|valid_people==expected_dev and
              train_people.isdisjoint(test_people) and
              valid_people.isdisjoint(test_people) and
              set(groups[train_idx])==train_people and
              set(groups[valid_idx])==valid_people, 'Inner participant partition mismatch')
        check(set(record['outer_test_participants'].split(';'))==set(test_people) and
              int(record['n_train_trials'])==len(train_idx) and
              int(record['n_validation_trials'])==len(valid_idx),
              'Saved trial counts or outer-test labels mismatch')
        inner_indices.append((train_idx,valid_idx))
        validation_rows.extend(valid_idx.tolist())
        verified_inner.append(record)
    check(sorted(validation_rows)==sorted(dev_idx.tolist()),
          'Each development trial must appear once in inner validation')

    for procedure,candidates in (('C+B',CB_CANDIDATES),('C+B+G',CBG_CANDIDATES)):
        fold_scores=[]
        for candidate in candidates:
            X=candidate_matrix(candidate)
            squared_errors=[]
            for inner_number,(tr,va) in enumerate(inner_indices,1):
                fitted=candidate_pipeline(candidate)
                tic=time.perf_counter()
                fitted.fit(X[tr],y[tr])
                fit_seconds=time.perf_counter()-tic
                assert_fitted_preprocessing(fitted,candidate,X[tr])
                tic=time.perf_counter()
                predicted=fitted.predict(X[va])
                prediction_seconds=time.perf_counter()-tic
                check(len(predicted)==len(va) and np.isfinite(predicted).all(),
                      'Invalid inner-validation predictions')
                squared_errors.extend(((predicted-y[va])**2).tolist())
                inner_scores.append({
                    'outer_fold':fold,'inner_fold':inner_number,**candidate,
                    'n_train_trials':len(tr),'n_validation_trials':len(va),
                    'train_RMSE_W_per_kg':rmse(y[tr],fitted.predict(X[tr])),
                    'validation_RMSE_W_per_kg':rmse(y[va],predicted),
                    'validation_SSE_W2_per_kg2':float(np.sum((predicted-y[va])**2,dtype=np.float64)),
                    'fit_seconds':fit_seconds,'prediction_seconds':prediction_seconds,
                })
            check(len(squared_errors)==len(dev_idx), 'Incomplete candidate validation')
            score=float(np.sqrt(np.mean(squared_errors,dtype=np.float64)))
            fold_scores.append({'outer_fold':fold,**candidate,
                                'inner_pooled_RMSE_W_per_kg':score})
        pooled_scores.extend(fold_scores)
        chosen=choose(fold_scores,procedure)
        selections.append({'outer_fold':fold,**chosen})

        X=candidate_matrix(chosen)
        model=candidate_pipeline(chosen)
        tic=time.perf_counter()
        model.fit(X[dev_idx],y[dev_idx])
        fit_seconds=time.perf_counter()-tic
        assert_fitted_preprocessing(model,chosen,X[dev_idx])
        tic=time.perf_counter()
        predicted=model.predict(X[test_idx])
        predict_seconds=time.perf_counter()-tic
        check(len(predicted)==len(test_idx) and np.isfinite(predicted).all(),
              'Invalid outer-held-out predictions')
        timings.append({'outer_fold':fold,'procedure':procedure,
                        'representation':chosen['representation'],
                        'alpha':chosen['alpha'],'n_components':chosen['n_components'],
                        'n_features':chosen['n_features'],
                        'fit_seconds':fit_seconds,'prediction_seconds':predict_seconds,
                        'prediction_seconds_per_trial':predict_seconds/len(test_idx),
                        'n_train_trials':len(dev_idx),'n_predicted_trials':len(test_idx)})
        for index,value in zip(test_idx,predicted):
            row=rows[index]
            new_predictions.append({'procedure':procedure,'outer_fold':fold,
                                    'subject_id':row['subject_id'],'trial_id':row['trial_id'],
                                    'trial_uid':row['trial_uid'],
                                    'observed_W_per_kg':float(y[index]),
                                    'predicted_W_per_kg':float(value)})
        print(f'Outer {fold}: {procedure:5s} -> {chosen["representation"]}, '
              f'alpha={chosen["alpha"]}' +
              (f', PCA={chosen["n_components"]}' if chosen['n_components'] else ''))

elapsed_seconds=time.perf_counter()-t0
print(f'Finished {len(new_predictions)} new predictions in {elapsed_seconds:.1f} s')

In [ ]:
# 7 — Verify matched predictions and independently recalculate primary metrics
checks=[]
def verify(name,condition):
    checks.append({'check':name,'status':'PASS' if bool(condition) else 'FAIL'})
    check(condition,name)

verify('source_and_previous_archive_fingerprints_match',
       source_sha==summary04['source_sha256']==summary05['source_sha256'] and
       summary05['reference_notebook04_zip_sha256']==sha04)
verify('biological_csv_has_21_unique_valid_profiles',len(biological)==21 and
       set(biological)==set(FOLD_OF))
verify('biological_join_one_to_one_by_subject_id',len(join_manifest)==21 and
       all(r['join_status']=='PASS' for r in join_manifest))
verify('source_284_trials_60757_cycles',len(rows)==284 and
       sum(r['retained_cycles'] for r in rows)==60757)
verify('fixed_outer_folds_match',all(sum(r['outer_fold']==f for r in rows)==count
       for f,count in zip(OUTER_FOLDS,EXPECTED_COUNTS)))
verify('all_20_saved_inner_folds_reused',len(verified_inner)==20 and
       verified_inner==inner04)
verify('six_CB_and_36_CBG_candidates',len(CB_CANDIDATES)==6 and
       len(CBG_CANDIDATES)==36)
verify('840_inner_candidate_fit_records',len(inner_scores)==5*4*(6+36))
verify('210_pooled_candidate_scores',len(pooled_scores)==5*(6+36))
verify('ten_selected_complete_models',len(selections)==10)
for row in selections:
    scores=[r for r in pooled_scores if
            r['outer_fold']==row['outer_fold'] and r['procedure']==row['procedure']]
    result=choose(scores,row['procedure'])
    check(all(row[key]==result[key] for key in
              ('representation','alpha','n_components','n_features',
               'inner_pooled_RMSE_W_per_kg')),
          f'Invalid selection at outer {row["outer_fold"]}: {row["procedure"]}')
verify('all_selected_configs_minimise_pooled_inner_RMSE',True)

# Recalculate all pooled selection scores from saved inner SSE/count rows.
for item in pooled_scores:
    matching=[r for r in inner_scores if r['outer_fold']==item['outer_fold']
              and r['procedure']==item['procedure']
              and r['representation']==item['representation']
              and r['alpha']==item['alpha']
              and r['n_components']==item['n_components']]
    check(len(matching)==4, 'Candidate lacks four inner-validation records')
    score=np.sqrt(sum(r['validation_SSE_W2_per_kg2'] for r in matching)
                  /sum(r['n_validation_trials'] for r in matching))
    check(abs(score-item['inner_pooled_RMSE_W_per_kg'])<1e-12,
          'Pooled selection score differs from saved fold errors')
verify('pooled_candidate_scores_recomputed_from_validation_SSE',True)

ledger=[]
for procedure in ('C','C+G'):
    for row in rows:
        old=prior_predictions[procedure][row['trial_uid']]
        ledger.append({'procedure':procedure,'outer_fold':row['outer_fold'],
                       'subject_id':row['subject_id'],'trial_id':row['trial_id'],
                       'trial_uid':row['trial_uid'],
                       'observed_W_per_kg':row['observed_W_per_kg'],
                       'predicted_W_per_kg':float(old['predicted_W_per_kg'])})
ledger.extend(new_predictions)
verify('exactly_1136_prediction_rows',len(ledger)==4*284)
verify('one_finite_prediction_per_procedure_and_trial',
       len({(r['procedure'],r['trial_uid']) for r in ledger})==len(ledger) and
       all(np.isfinite(r['predicted_W_per_kg']) for r in ledger))
# Compare numeric targets because earlier ledger fields were read from CSV.
verify('matched_284_UIDs_and_targets_across_four_procedures',
       all({r['trial_uid'] for r in ledger if r['procedure']==p}==set(uids)
           for p in ('C','C+G','C+B','C+B+G')) and
       all(float(r['observed_W_per_kg'])==float(prior_ledger[r['trial_uid']]['observed_W_per_kg'])
           for r in ledger))
verify('C_and_CG_predictions_reused_exactly',
       all(r['predicted_W_per_kg']==float(prior_predictions[r['procedure']][r['trial_uid']]['predicted_W_per_kg'])
           for r in ledger if r['procedure'] in ('C','C+G')))

pooled_metrics,fold_metrics=[],[]
for procedure in ('C','C+G','C+B','C+B+G'):
    selected=[r for r in ledger if r['procedure']==procedure]
    pooled_metrics.append({'procedure':procedure,**metrics(
        [r['observed_W_per_kg'] for r in selected],
        [r['predicted_W_per_kg'] for r in selected])})
    for fold in OUTER_FOLDS:
        part=[r for r in selected if r['outer_fold']==fold]
        fold_metrics.append({'outer_fold':fold,'procedure':procedure,**metrics(
            [r['observed_W_per_kg'] for r in part],
            [r['predicted_W_per_kg'] for r in part])})
verify('all_20_outer_fold_metric_rows',len(fold_metrics)==20)
verify('all_pooled_metrics_finite',all(
       np.isfinite(r['RMSE_W_per_kg']) and np.isfinite(r['MAE_W_per_kg']) and
       np.isfinite(r['bias_W_per_kg']) and np.isfinite(r['R2'])
       for r in pooled_metrics))
for name in ('C','C+G'):
    now=next(r for r in pooled_metrics if r['procedure']==name)
    old=next(r for r in previous_metrics05 if r['procedure']==name)
    for key in ('RMSE_W_per_kg','MAE_W_per_kg','bias_W_per_kg','R2'):
        check(abs(now[key]-float(old[key]))<1e-12,
              f'Reused {name} score differs from Notebook 05')
verify('prior_C_and_CG_pooled_metrics_unchanged',True)

result={r['procedure']:r for r in pooled_metrics}
delta_primary=result['C+B']['RMSE_W_per_kg']-result['C+B+G']['RMSE_W_per_kg']
delta_support=result['C+G']['RMSE_W_per_kg']-result['C+B+G']['RMSE_W_per_kg']
delta_biology=result['C']['RMSE_W_per_kg']-result['C+B']['RMSE_W_per_kg']
print('\nPooled RMSE (W/kg):')
for r in pooled_metrics:
    print(f'  {r["procedure"]:5s}: {r["RMSE_W_per_kg"]:.6f}')
print(f'Primary C+B − C+B+G: {delta_primary:+.6f} W/kg')
print(f'Supporting C+G − C+B+G: {delta_support:+.6f} W/kg')
print(f'Descriptive C − C+B: {delta_biology:+.6f} W/kg')
print(f'All {len(checks)} final checks passed; participant-cluster intervals remain for E5.2.')

In [ ]:
# 8 — Export reusable verification and prediction evidence (no raw gait or age/height values)
OUT.mkdir(parents=True,exist_ok=True)
notebook_filename='06_Bacek_Biological_Information_Comparison.ipynb'
notebook_code_sha=None
if Path(notebook_filename).is_file():
    doc=json.loads(Path(notebook_filename).read_text(encoding='utf-8'))
    joined=''.join(''.join(cell['source']) for cell in doc['cells']
                   if cell['cell_type']=='code')
    notebook_code_sha=hashlib.sha256(joined.encode('utf-8')).hexdigest()

write_csv(OUT/'eligible_trial_ledger.csv',rows,
          ['trial_uid','subject_id','trial_id','outer_fold','retained_cycles',
           'speed_m_s','relative_cadence','condition_index','observed_W_per_kg'])
write_csv(OUT/'biological_join_manifest.csv',join_manifest,
          ['subject_id','join_status','age_present_and_positive','height_present_and_positive'])
write_csv(OUT/'inner_fold_manifest.csv',inner04,
          ['outer_fold','inner_fold','train_participants','validation_participants',
           'outer_test_participants','n_train_trials','n_validation_trials'])
write_csv(OUT/'inner_candidate_scores.csv',inner_scores,
          ['outer_fold','inner_fold','procedure','representation','alpha','n_components',
           'n_features','n_train_trials','n_validation_trials','train_RMSE_W_per_kg',
           'validation_RMSE_W_per_kg','validation_SSE_W2_per_kg2','fit_seconds',
           'prediction_seconds'])
write_csv(OUT/'pooled_candidate_scores.csv',pooled_scores,
          ['outer_fold','procedure','representation','alpha','n_components',
           'n_features','inner_pooled_RMSE_W_per_kg'])
write_csv(OUT/'selected_models.csv',selections,
          ['outer_fold','procedure','representation','alpha','n_components',
           'n_features','inner_pooled_RMSE_W_per_kg'])
write_csv(OUT/'held_out_predictions.csv',ledger,
          ['procedure','outer_fold','subject_id','trial_id','trial_uid',
           'observed_W_per_kg','predicted_W_per_kg'])
write_csv(OUT/'pooled_metrics.csv',pooled_metrics,
          ['procedure','n_trials','RMSE_W_per_kg','MAE_W_per_kg','bias_W_per_kg','R2'])
write_csv(OUT/'outer_fold_metrics.csv',fold_metrics,
          ['outer_fold','procedure','n_trials','RMSE_W_per_kg','MAE_W_per_kg',
           'bias_W_per_kg','R2'])
write_csv(OUT/'outer_model_timings.csv',timings,
          ['outer_fold','procedure','representation','alpha','n_components','n_features',
           'fit_seconds','prediction_seconds','prediction_seconds_per_trial',
           'n_train_trials','n_predicted_trials'])
write_csv(OUT/'verification_checks.csv',checks,['check','status'])

summary={
    'status':'PASS' if all(r['status']=='PASS' for r in checks) else 'REVIEW_REQUIRED',
    'experiment':'E2.2',
    'prediction_setting':'new-person, internally held-out and exploratory-informed',
    'source_hdf5':H5.name,'source_sha256':source_sha,
    'biological_csv':BIO_CSV.name,'biological_csv_sha256':bio_sha,
    'notebook04_zip':REFERENCE04.name,'notebook04_zip_sha256':sha04,
    'notebook05_zip':REFERENCE05.name,'notebook05_zip_sha256':sha05,
    'notebook04_code_sha256':summary04.get('notebook_code_sha256'),
    'notebook05_code_sha256':summary05.get('notebook_code_sha256'),
    'target':'stored corrected trial-level metcost_mean, net W/kg',
    'biological_fields':['age_years','height_m'],
    'ignored_biological_fields':'all fields other than subject_id, age_years and height_m',
    'n_primary_trials':284,'n_primary_participants':21,
    'n_biological_profiles':21,
    'C_and_CG':'reused without refitting from Notebook 05',
    'C_plus_B':'15 unscaled condition cells + two train-scaled biological columns + Ridge',
    'C_plus_B_plus_G':'C+B + separately train-scaled gait W/D/P, PCA applied to gait only; Ridge',
    'n_candidates':{'C+B':6,'C+B+G':36},
    'alpha_grid':list(ALPHAS),'pca_components':list(PCA_COUNTS),
    'pca':{'whiten':False,'random_state':2026},
    'inner_splitter':'exact 20 four-way participant-grouped folds copied from Notebook 04/05',
    'tie_rule_CB':'exact pooled inner RMSE, then smaller alpha',
    'tie_rule_CBG':'exact pooled inner RMSE, fewer total features, representation ID, components, alpha',
    'primary_comparison':'RMSE(C+B) - RMSE(C+B+G)',
    'primary_delta_RMSE_W_per_kg':delta_primary,
    'supporting_delta_CG_vs_CBG_RMSE_W_per_kg':delta_support,
    'descriptive_delta_C_vs_CB_RMSE_W_per_kg':delta_biology,
    'uncertainty_interval_included':False,
    'started_at_utc':started_at,
    'finished_at_utc':datetime.now(timezone.utc).isoformat(),
    'elapsed_seconds':elapsed_seconds,
    'software':{'python':platform.python_version(),'numpy':np.__version__,
                'scipy':scipy.__version__,'scikit_learn':sklearn.__version__,
                'h5py':h5py.__version__,'os':platform.platform(),
                'cpu_count':os.cpu_count(),'kernel_python_executable':sys.executable},
    'notebook_filename':notebook_filename,
    'notebook_code_sha256':notebook_code_sha,
    'code_fingerprint_basis':'concatenated notebook code-cell sources, UTF-8',
    'verification':{r['check']:r['status'] for r in checks},
}
(OUT/'run_summary.json').write_text(json.dumps(summary,indent=2,allow_nan=False),
                                    encoding='utf-8')
filenames=('eligible_trial_ledger.csv','biological_join_manifest.csv',
           'inner_fold_manifest.csv','inner_candidate_scores.csv',
           'pooled_candidate_scores.csv','selected_models.csv',
           'held_out_predictions.csv','pooled_metrics.csv','outer_fold_metrics.csv',
           'outer_model_timings.csv','verification_checks.csv','run_summary.json')
with zipfile.ZipFile(ARCHIVE,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for filename in filenames:
        z.write(OUT/filename,arcname=filename)
print('Saved:',ARCHIVE,'| evidence files:',len(filenames),
      '| status:',summary['status'])